# exp057_xgb_catboost_pf_confidence_only_features train

Train-side audit for XGBoost / CatBoost residual models with PF/Beam confidence-only features.


## Contents

1. Setup and configuration
2. Input artifact check
3. PF confidence model audit
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from pathlib import Path
import json
import pandas as pd

from settings import ExperimentPaths, load_config
from pf_confidence_model_audit import resolve_feature_path, run_audit, get_nested


In [ ]:
paths = ExperimentPaths()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", config["experiment"]["name"])
print("Route:", config["experiment"]["route"])
print("Parent:", config["lineage"]["parent"])
print("Supporting artifact:", config["lineage"]["supporting_artifact"])
print("Base estimator:", get_nested(config, "model.estimator"))
print("Variants:", [item["name"] for item in get_nested(config, "model.variants", [])])


## 2. Input artifact check


In [ ]:
feature_path = resolve_feature_path(paths, get_nested(config, "data.feature_path"))
print("Feature path:", feature_path)
print("Exists:", feature_path.exists())

preview = pd.read_csv(feature_path, nrows=5)
print("Rows preview:", len(preview))
print("Columns:", list(preview.columns))


## 3. PF confidence model audit


In [ ]:
summary = run_audit(
    paths,
    config,
    feature_path,
    output_dir=paths.artifacts_dir,
    max_wells=None,
    max_train_rows_override=None,
    skip_exp026_control=False,
)
print(json.dumps(summary, indent=2))


## 4. Metrics and artifacts


In [ ]:
metrics = pd.read_csv(paths.artifacts_dir / "pf_confidence_metrics.csv")
buckets = pd.read_csv(paths.artifacts_dir / "pf_confidence_bucket_metrics.csv")
splits = pd.read_csv(paths.artifacts_dir / "pf_confidence_split_metrics.csv")
family_matrix = pd.read_csv(paths.artifacts_dir / "pf_confidence_family_matrix.csv")
feature_parity = pd.read_csv(paths.artifacts_dir / "pf_confidence_feature_parity_report.csv")

display(metrics.sort_values(["audit", "rmse"]).head(30))
display(family_matrix.sort_values(["audit", "rmse"]).head(30))
display(feature_parity.sort_values(["source", "feature"]).head(60))
display(buckets.sort_values(["audit", "candidate", "bucket"]).head(30))
display(splits.sort_values(["audit", "candidate", "split"]).head(30))
